# OpTC Host ML — LGBM, undersampled negatives (shared shards)

This notebook reads the same persisted 75/10/15 split and vocabulary as the other models.
Validation and test retain every event, exactly like the other notebooks.

Standard LightGBM `Dataset` construction still needs a full training representation resident in
memory. With ~3.48 billion train rows that is not feasible on a standard Kaggle runtime —
`data.materialize('train', ...)` will correctly refuse rather than silently truncate. Because the
positive label is extremely rare (201,551 of 3,476,277,622 train rows, ~0.0058%), this notebook
instead does random negative undersampling while streaming the full train split exactly once:
every positive row is kept, negatives are kept with a fixed probability. See the training cell
for the exact mechanism and the metrics report for exact counts kept.


Upload the archive created by package_host_shards.py as one dataset, then attach it.
Setup locates its manifest and maps all packaged volumes automatically. Dataset capacity is not notebook RAM.


In [ ]:
from pathlib import Path
import sys, json
# Set an explicit path here only if several Host input datasets are attached.
MANIFEST = Path(
    '/kaggle/input/datasets/nhatnguyen2101/'
    'optc-host-ml-full-input/manifest.json'
)
if MANIFEST is None:
    candidates = []
    for path in Path('/kaggle/input').rglob('manifest.json'):
        try:
            meta = json.loads(path.read_text())
        except (OSError, ValueError):
            continue
        if isinstance(meta, dict) and meta.get('format') == 'host_binary_delta_npz':
            candidates.append(path)
    if len(candidates) != 1:
        raise RuntimeError(f'Expected one Host dataset; set MANIFEST explicitly. Found: {candidates}')
    MANIFEST = candidates[0]
MANIFEST = Path(MANIFEST)
meta = json.loads(MANIFEST.read_text())
VOLUME_ROOTS = {key: MANIFEST.parent / 'volumes' / key for key in meta['volumes']}
sys.path.insert(0, str(MANIFEST.parent))
from host_shards import HostShardDataset
from shard_metrics import evaluate_stream
data = HostShardDataset(MANIFEST, VOLUME_ROOTS)
print('Manifest:', MANIFEST)
# Decode one shard per split to check the uploaded paths and format.
for split in ('train', 'val', 'test'):
    X_check, y_check = next(data.iter_batches(split))
    print(split, 'first shard:', X_check.shape, 'positive labels:', int(y_check.sum()))
    del X_check, y_check
print(json.dumps(data.meta['splits'], indent=2))
print('Full train CSR lower bound GiB:', data.csr_bytes('train') / 2**30)
OUTPUT = Path('/kaggle/working')
OUTPUT.mkdir(exist_ok=True)


In [ ]:
MODEL_NAME = 'lgbm'
import numpy as np
from scipy import sparse
import lightgbm as lgb

# Every positive row is kept; negatives are kept with probability `keep_p`,
# streamed shard-by-shard so peak memory stays at one shard, never the full
# 3.48B-row split. This is a uniform random subsample of the negative
# population across the *entire* stream (not a byte-budget truncation like
# `data.materialize(max_bytes=...)`, which just stops at whichever shards
# come first in the manifest and can end up with few or no positives).
NEG_PER_POS = 30          # negatives kept per positive, on expectation
SEED = 1
rng = np.random.default_rng(SEED)

label0 = data.meta['splits']['train']['label_0']
label1 = data.meta['splits']['train']['label_1']
keep_p = min(1.0, NEG_PER_POS * label1 / label0)
print(f'target negative keep-probability: {keep_p:.6f}')

X_parts, y_parts = [], []
n_pos = n_neg_seen = n_neg_kept = 0
for X_shard, y_shard in data.iter_batches('train'):
    pos = y_shard == 1
    if pos.any():
        X_parts.append(X_shard[pos]); y_parts.append(y_shard[pos])
        n_pos += int(pos.sum())
    neg_idx = np.flatnonzero(~pos)
    n_neg_seen += neg_idx.size
    if neg_idx.size:
        keep = neg_idx[rng.random(neg_idx.size) < keep_p]
        if keep.size:
            X_parts.append(X_shard[keep]); y_parts.append(y_shard[keep])
            n_neg_kept += keep.size

X_train = sparse.vstack(X_parts, format='csr')
y_train = np.concatenate(y_parts)
del X_parts, y_parts
print(f'kept {n_pos} positives, {n_neg_kept}/{n_neg_seen} negatives -> X_train {X_train.shape}')

train = lgb.Dataset(X_train, label=y_train, free_raw_data=True)
# is_unbalance: even after undersampling, the kept ratio (~NEG_PER_POS:1) is
# still far from 1:1, so let LightGBM reweight the loss on top of that.
model = lgb.train({'objective':'binary', 'metric':'None', 'num_leaves':31,
    'learning_rate':0.05, 'seed':SEED, 'num_threads':4, 'verbosity':-1,
    'is_unbalance': True}, train, num_boost_round=300)
del X_train, y_train, train
predict = lambda X: model.predict(X)
model.save_model(str(OUTPUT / 'lgbm_model.txt'))


In [ ]:
validation = evaluate_stream(data, 'val', predict)
test = evaluate_stream(data, 'test', predict, threshold=validation['threshold'])
report = {'model': MODEL_NAME, 'split_sha256':data.meta['split_sha256'],
    'vocabulary_sha256':data.meta['vocabulary_sha256'], 'seed':SEED,
    'train_sampling': {'method':'random_negative_undersampling',
        'neg_per_pos_target':NEG_PER_POS, 'positives_kept':n_pos,
        'negatives_kept':n_neg_kept, 'negatives_seen':n_neg_seen,
        'rows_kept':n_pos + n_neg_kept},
    'validation':validation, 'test':test}
(OUTPUT / (MODEL_NAME + '_metrics.json')).write_text(json.dumps(report, indent=2))
print(json.dumps(report, indent=2))
